# Combine all datasets and Train Test Spilt

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import numpy as np
sys.path.append('../../../')   # Add parent directory to Python path
import pickle
from utils.preprocessing import *
from utils.segmentation import *
from utils.visualization import *

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split

np.random.seed(42)  # For reproducibility


## 1. Combine all datasets

### 7 classes

In [5]:
# Load the curb data (which appears to be stored as a dictionary with scene_0 and scene_1)
with open('../../../data/Training/1s_30hz/curb_1s_combined_all.pkl', 'rb') as f:
    curb_data = pickle.load(f)
    data_curb_0 = curb_data['scene_0']
    data_curb_1 = curb_data['scene_1']
# Load the other surface types (these appear to be stored as arrays)
with open('../../../data/Training/1s_30hz/asphalt_1s_combined_all.pkl', 'rb') as f:
    data_asphalt = pickle.load(f)

with open('../../../data/Training/1s_30hz/cobblestone_1s_combined_all.pkl', 'rb') as f:
    data_cobblestone = pickle.load(f)

with open('../../../data/Training/1s_30hz/compactgravel_1s_combined_all.pkl', 'rb') as f:
    data_compact_gravel = pickle.load(f)

with open('../../../data/Training/1s_30hz/dirt_1s_combined_all.pkl', 'rb') as f:
    data_Dirt = pickle.load(f)

with open('../../../data/Training/1s_30hz/pavingstone_1s_combined_all.pkl', 'rb') as f:
    data_PavingStone = pickle.load(f)

# Print shapes to verify the data was loaded correctly
print("Curb (scene 0):", data_curb_0.shape)
print("Curb (scene 1):", data_curb_1.shape)
print("Asphalt:", data_asphalt.shape)
print("Cobblestone:", data_cobblestone.shape)
print("Compact Gravel:", data_compact_gravel.shape)
print("Dirt:", data_Dirt.shape)
print("Paving Stone:", data_PavingStone.shape)


Curb (scene 0): (12267, 30, 3)
Curb (scene 1): (617, 30, 3)
Asphalt: (704, 30, 3)
Cobblestone: (486, 30, 3)
Compact Gravel: (479, 30, 3)
Dirt: (577, 30, 3)
Paving Stone: (645, 30, 3)


In [8]:
# randomly select 617 samples of curb_scene_0 to balance the dataset
data_curb_0_balanced = select_random_samples(data_curb_0, data_curb_1.shape[0])
data_curb_0_balanced.shape


(617, 30, 3)

In [9]:
# Add labels
# Assign labels for each surface type
datasets = [
    (data_curb_0_balanced, "curb_0"),
    (data_curb_1, "curb_1"),
    (data_asphalt, "asphalt"),
    (data_cobblestone, "cobblestone"),
    (data_compact_gravel, "compact_gravel"),
    (data_Dirt, "dirt"),
    (data_PavingStone, "paving_stone")
]

# Combine all into a single list of (sensor_values, label)
combined_dataset = []
for data, label in datasets:
    for segment in data:
        combined_dataset.append((segment, label))

## 2. Train Test Spilt


In [10]:
# 80% train, 20% test
train_set, test_set = train_test_split(combined_dataset, test_size=0.2, random_state=42, shuffle=True, stratify=[label for data, label in combined_dataset])
print(len(train_set), len(test_set))
# Separate sensor values and labels
X_train = [x for x, _ in train_set]
y_train = [label for _, label in train_set]
X_test = [x for x, _ in test_set]
y_test = [label for _, label in test_set]
print(f"Label train: {len(y_train)}, Label test: {len(y_test)}")

3300 825
Label train: 3300, Label test: 825


## 3. Normalise dataset

In [12]:
X_train_normalized = normalize_3d_data(X_train)
X_test_normalized = normalize_3d_data(X_test)

# 4. Labels from string to integer

In [13]:
label_encoder = LabelEncoder()
y_train_int = label_encoder.fit_transform(y_train)
y_test_int = label_encoder.transform(y_test)

print("Classes:", label_encoder.classes_)
print("First 10 y_train_int:", y_train_int[:10])
print("First 10 y_test_int:", y_test_int[:10])
for idx, label in enumerate(label_encoder.classes_):
    print(f"{idx}: {label}")


Classes: ['asphalt' 'cobblestone' 'compact_gravel' 'curb_0' 'curb_1' 'dirt'
 'paving_stone']
First 10 y_train_int: [0 4 0 2 2 5 6 0 3 2]
First 10 y_test_int: [3 5 0 0 6 5 0 1 2 6]
0: asphalt
1: cobblestone
2: compact_gravel
3: curb_0
4: curb_1
5: dirt
6: paving_stone


## 5: One-hot encode the labels

In [14]:
y_train_onehot = to_categorical(y_train_int)
y_test_onehot = to_categorical(y_test_int)

print(y_train_onehot.shape)
print(y_test_onehot.shape)

(3300, 7)
(825, 7)


In [15]:
# Randomly select an index and check that the one-hot encoding matches the original label
r = np.random.randint(len(y_train_int))
assert y_train_onehot[r].argmax() == y_train_int[r]
r = np.random.randint(len(y_test_int))
assert y_test_onehot[r].argmax() == y_test_int[r]

## 6. Save train, test data and labels

In [16]:
# Save test data
with open('../../../data/Training/1s_30hz/7class_balanced/X_test_data.pkl', 'wb') as f:
    pickle.dump(X_test_normalized, f)
with open('../../../data/Training/1s_30hz/7class_balanced/y_test_onehot.pkl', 'wb') as f:
    pickle.dump(y_test_onehot, f)

## 6. Train, validation Spilt

In [17]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_normalized,
    y_train_onehot,
    test_size=0.2,           # 20% for validation
    random_state=42,
    shuffle=True
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)

Train shape: (2640, 30, 3) (2640, 7)
Validation shape: (660, 30, 3) (660, 7)


In [18]:
# Save training and validation data
with open('../../../data/Training/1s_30hz/7class_balanced/X_train_normalized.pkl', 'wb') as f:
    pickle.dump(X_train, f)

with open('../../../data/Training/1s_30hz/7class_balanced/X_val_normalized.pkl', 'wb') as f:
    pickle.dump(X_val, f)

with open('../../../data/Training/1s_30hz/7class_balanced/y_train_onehot.pkl', 'wb') as f:
    pickle.dump(y_train, f)

with open('../../../data/Training/1s_30hz/7class_balanced/y_val_onehot.pkl', 'wb') as f:
    pickle.dump(y_val, f)